# CLM-001-R3b-CF-001：只读反事实诊断
在原私有 bingjiangli/minicells 导入本文件，开启 Internet，选择 GPU，启用已有 KAGGLE_API_TOKEN 与 GITHUB_TOKEN，然后 Save & Run All。无需切换模式或填写哈希。
自动：旧59项回读 → 原57索引核验 → Calibration固定工作量 → 原Greedy一致性 → CF2/CF3/CF1 → 汇总回传。没有训练，没有HF写入，Confirmation保持封闭。
新原始结果只保存在私有Output；后续同一Notebook运行自动回读已保存Output并续接。

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
from kaggle_secrets import UserSecretsClient

ORCHESTRATOR_SHA = "c2795e831037c2900fb1a524ea6b3edf6d6c0382"
ENGINE_SHA = "ed2accbc90f541325ca2f6c53229e9c81475768d"
KAGGLE_CLI_SHA = "6894864ff4095cf98e9e36d3a694ff5fc9ba3818"
BASE = Path("/kaggle/temp/minicells-r3b-cf001")
CODE = BASE / ("orchestrator-" + ORCHESTRATOR_SHA[:12])
ENGINE = BASE / ("engine-" + ENGINE_SHA[:12])
OUTPUT = Path("/kaggle/working/minicells-r3b-cf001") / ORCHESTRATOR_SHA[:12]
RELATIVE = Path(
    "research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3b"
)
os.environ.update(
    OMP_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
    TOKENIZERS_PARALLELISM="false",
    HF_HUB_DISABLE_PROGRESS_BARS="1",
)


def run(args, cwd=None):
    print("[CF-001] " + " ".join(map(str, args)), flush=True)
    return subprocess.run(list(map(str, args)), cwd=cwd, check=True)


secrets = UserSecretsClient()
for name in ("KAGGLE_API_TOKEN", "GITHUB_TOKEN"):
    if not os.environ.get(name, "").strip():
        try:
            os.environ[name] = secrets.get_secret(name).strip()
        except Exception:
            raise RuntimeError("Enable existing Kaggle Secret " + name) from None
    if not os.environ[name]:
        raise RuntimeError("Missing existing Kaggle Secret " + name)
patterns = [
    "/pyproject.toml",
    "/src/",
    "/shared/python/",
    "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/",
    "/research/projects/cellular-language-models/engineering/kaggle-continuation/",
    "/research/projects/cellular-language-models/experiments/006-path-first-clm/",
    "/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/",
    "/research/projects/continual-learning/context/04-continual-learning/scripts/research/",
    "/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/",
]
for checkout, sha in ((CODE, ORCHESTRATOR_SHA), (ENGINE, ENGINE_SHA)):
    if sha.startswith("__"):
        raise RuntimeError("Notebook implementation has not been pinned")
    if not (checkout / ".git").exists():
        checkout.mkdir(parents=True, exist_ok=False)
        run(["git", "init", "--quiet"], checkout)
        run(
            ["git", "remote", "add", "origin", "https://github.com/ArcheLabs/mini-cells.git"],
            checkout,
        )
        run(["git", "sparse-checkout", "init", "--no-cone"], checkout)
        run(["git", "sparse-checkout", "set", "--no-cone", *patterns], checkout)
        run(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", sha], checkout)
        run(["git", "checkout", "--detach", "FETCH_HEAD"], checkout)
    if (
        subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=checkout, text=True).strip()
        != sha
    ):
        raise RuntimeError("SOURCE_SHA_MISMATCH")
# Versioned Output support and its v<N> label fix are pinned to the audited official source.
run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "kaggle @ git+https://github.com/Kaggle/kaggle-cli.git@" + KAGGLE_CLI_SHA,
    ]
)
for module, requirement in (("tokenizers", "tokenizers"), ("matplotlib", "matplotlib")):
    if importlib.util.find_spec(module) is None:
        run([sys.executable, "-m", "pip", "install", "--quiet", requirement])
run(["kaggle", "--version"])
print(
    {
        "ACTION": "CF-001-READ-ONLY-COUNTERFACTUAL",
        "MEASUREMENT_ENGINE": ENGINE_SHA,
        "EXECUTION_ORCHESTRATOR": ORCHESTRATOR_SHA,
        "SOURCE_CANDIDATE_VERSION": 65,
        "HF_WRITES": 0,
        "TRAINING_UPDATES": 0,
        "CONFIRMATION": "SEALED",
    },
    flush=True,
)
run(
    [
        sys.executable,
        CODE / RELATIVE / "counterfactual-stop-route/run.py",
        "--engine-root",
        ENGINE,
        "--orchestrator-root",
        CODE,
        "--output",
        OUTPUT,
        "--cache",
        BASE / "historical-output-cache",
        "--device",
        "cuda:0",
        "--segment-seconds",
        "6900",
        "--publish-github",
    ],
    CODE,
)
print(
    "OUTPUT_VERSION_PENDING_SAVE: this Save & Run All must retain Output. A later launch verifies saved results and resumes automatically.",
    flush=True,
)